# S&P 500 momentum shortlist

Which S&P 500 names should be on our momentum watchlist? Give me the top 20 above their 200-day average, ranked by 12–1 month momentum.

**Data and assumptions**

- Current S&P 500 membership snapshot, with daily prices through 2026-10-16; this is a current-universe screen, not a historical membership backtest.
- Use split-adjusted closes for price returns and compatible price/share units for dollar volume. The vendor daily feed does not verify exact regular-session or auction prints.

**Requested answer:** shortlist (table), screening_coverage (table).

This notebook contains its complete request and setup. Share this file; no local case JSON or helper module is required. It contains no pre-filled market results.

## Connect

Select a Python kernel with `epoch-sdk[notebook]` installed. On the prepared
server, choose **Epoch SDK (local prototype)**.

The notebook needs only your Epoch login and the two service addresses. The
collapsed helper below obtains a current identity token from the environment's
configured Epoch login command, or uses `EPOCH_TOKEN`. It prints no credentials
and reads no service configuration files. On another machine, supply your own
`EPOCH_TOKEN`, `EPOCH_BACKEND_URL` and `EPOCH_GRAPH_URL`.

The next cell prepares the client and checks both services. It starts no research.

In [ ]:
"""Obtain only an Epoch bearer token from the local operator's login provider."""

from __future__ import annotations

import json
import os
import subprocess


def identity_token() -> str:
    """Read a static token or ask the configured login helper for a current one.

    The prepared notebook launcher configures a trusted argv list in the
    environment. It uses the owner's normal Epoch login/refresh flow. No shell
    is used, and neither provider output nor credentials appear in exceptions.
    Standalone notebook environments can continue to supply EPOCH_TOKEN alone.
    """
    command = os.environ.get("EPOCH_NOTEBOOK_TOKEN_COMMAND")
    if command:
        try:
            argv = json.loads(command)
        except (TypeError, ValueError):
            raise RuntimeError(
                "The notebook's Epoch login command is invalid."
            ) from None
        if (
            not isinstance(argv, list)
            or not argv
            or not all(isinstance(value, str) and value for value in argv)
        ):
            raise RuntimeError(
                "The notebook's Epoch login command must be a JSON argv list."
            )
        try:
            result = subprocess.run(
                argv, capture_output=True, text=True, timeout=30, check=False
            )
        except (OSError, subprocess.TimeoutExpired):
            raise RuntimeError(
                "Could not contact the local Epoch login helper. Restore the normal "
                "Epoch login and rerun this setup cell."
            ) from None
        token = result.stdout.strip()
        if result.returncode or not token or any(value.isspace() for value in token):
            raise RuntimeError(
                "The Epoch login needs attention. Sign in through the normal login "
                "flow, then rerun this setup cell."
            )
        return token
    token = os.environ.get("EPOCH_TOKEN")
    if not token:
        raise RuntimeError(
            "Set EPOCH_TOKEN to your Epoch identity token, or use the prepared local "
            "notebook server, which supplies Epoch login to its kernels."
        )
    return token

In [ ]:
import importlib.metadata
import httpx
from IPython.display import JSON, display
from epoch_sdk import RunRequest
from epoch_sdk.research import EpochClient, ResearchCompleted

# Connect to Epoch using your login, not vendor or model credentials.
_epoch_token = identity_token()
epoch = EpochClient.connect(
    graph_url=os.environ["EPOCH_GRAPH_URL"],
    backend_url=os.environ["EPOCH_BACKEND_URL"],
    token=_epoch_token,
)

print("Epoch SDK:", importlib.metadata.version("epoch-sdk"))
print("Native SDK:", importlib.metadata.version("stratifyx"))
async with httpx.AsyncClient(
    headers={"Authorization": "Bearer " + _epoch_token}, timeout=20,
) as health:
    for name, origin, route in [
        ("Backend", os.environ["EPOCH_BACKEND_URL"], "/api/v1/health"),
        ("EpochAI", os.environ["EPOCH_GRAPH_URL"], "/health/custom"),
    ]:
        response = await health.get(origin.rstrip("/") + route)
        response.raise_for_status()
        print(name + ": ready")
del _epoch_token

## The request

The task and context are the desk request shown above. The collapsed output
schema declares the requested fields, formats and units; expand it to inspect
or change them. It contains no result values.

Keep the attempt unchanged to reconnect, including after a kernel restart.
Choose a new attempt when intentionally changing the request or refreshing data.

In [ ]:
OUTPUTS = [{'name': 'shortlist',
  'type': 'table',
  'description': 'Ranked current-universe shortlist; one row per ticker, ordered by rank.',
  'schema': [{'name': 'rank',
              'type': 'int64',
              'description': 'Sequential rank starting at 1; descending momentum with ticker '
                             'ascending as the tie-break.'},
             {'name': 'ticker',
              'type': 'string',
              'description': 'Unique US-listed constituent ticker; the row label is a security, '
                             'not a timestamp.'},
             {'name': 'company',
              'type': 'string',
              'description': 'Company display name from the resolved asset metadata.'},
             {'name': 'sector',
              'type': 'string',
              'description': 'Sector label from the resolved asset metadata.'},
             {'name': 'as_of',
              'type': 'date',
              'description': 'Exact completed price session used by the screen: 2026-10-16.'},
             {'name': 'momentum_12_1',
              'type': 'float64',
              'description': 'Split-adjusted close 21 sessions ago / split-adjusted close 252 '
                             'sessions ago - 1.',
              'unit': 'fraction'},
             {'name': 'return_3m',
              'type': 'float64',
              'description': 'Split-adjusted close at as_of / close 63 sessions ago - 1; price '
                             'return.',
              'unit': 'fraction'},
             {'name': 'distance_to_sma_200',
              'type': 'float64',
              'description': 'As-of split-adjusted close / its trailing 200-session SMA - 1; '
                             'strictly positive.',
              'unit': 'fraction'},
             {'name': 'average_dollar_volume_20',
              'type': 'float64',
              'description': 'Mean of actual quoted close times traded shares over the last 20 '
                             'sessions, with price/volume on a consistent split basis.',
              'unit': 'USD/day'}]},
 {'name': 'screening_coverage',
  'type': 'table',
  'description': 'Exactly one coverage row for the captured roster and price cutoff.',
  'schema': [{'name': 'as_of',
              'type': 'date',
              'description': 'Price observation cutoff fixed for this case: 2026-10-16.'},
             {'name': 'membership_snapshot_at',
              'type': 'datetime',
              'description': 'Actual retrieval timestamp of the current S&P 500 roster; not a '
                             'claim of historical membership.'},
             {'name': 'universe_count',
              'type': 'int64',
              'description': 'Number of unique constituent tickers in the captured roster.'},
             {'name': 'excluded_missing_data_count',
              'type': 'int64',
              'description': 'Constituents excluded for missing required history, fields or '
                             'metadata.'},
             {'name': 'eligible_count',
              'type': 'int64',
              'description': 'Constituents with complete required data and a close strictly above '
                             'their 200-session SMA.'},
             {'name': 'returned_count',
              'type': 'int64',
              'description': 'min(20, eligible_count); do not pad the shortlist with failing '
                             'names.'}]}]

In [ ]:
TASK = 'Which S&P 500 names should be on our momentum watchlist? Give me the top 20 above their 200-day average, ranked by 12–1 month momentum.'
CONTEXT = ['Current S&P 500 membership snapshot, with daily prices through 2026-10-16; this is a '
 'current-universe screen, not a historical membership backtest.',
 'Use split-adjusted closes for price returns and compatible price/share units for dollar volume. '
 'The vendor daily feed does not verify exact regular-session or auction prints.']

PREFIX = os.environ.get("EPOCH_NOTEBOOK_PREFIX", "notebook-demo")
ATTEMPT = "desk-v2-01"
RESEARCH_ID = f"{PREFIX}-sp500_momentum_screener-{ATTEMPT}-research"
PARAMETERS = {'period': {'start': '2025-01-01', 'end': '2026-10-16'}}
request = RunRequest.model_validate({
    "task": TASK, "context": CONTEXT, "parameters": PARAMETERS, "outputs": OUTPUTS,
})
print("Research ID:", RESEARCH_ID)
display(JSON(request.to_wire(), expanded=False))

## Run the session

This is the cell that starts research or reconnects to the same operation. It
shows live activity and the published answer. If Epoch asks a question, choose
an option or write a reply and press **Send reply** inside the cell.

Nothing is selected automatically. If you leave a question waiting, rerun this
cell with the same research ID before continuing to the result cells.

In [ ]:
# The SDK owns streaming, questions and resume.
result = await epoch.run(
    request,
    operation_id=RESEARCH_ID,
)

## Use the result in Python

The session has already displayed the answer. These cells read its typed parts
for further analysis without printing the same narrative again. Table reads
retrieve every page; JSON and text are already in the saved answer.

In [ ]:
assert isinstance(result, ResearchCompleted), "Resume the session above first."
answer = result.response
json_answers = {part.name: part.value for part in answer.content.outputs if part.type == "json"}
text_answers = {part.name: part.value for part in answer.content.outputs if part.type == "text"}
# Read every page; a page size does not limit the total table size.
tables = {}
for part in result.response.content.outputs:
    if part.type != "table":
        continue
    rows = []
    async for page in epoch.iter_response_table(
        result.response.response_id,
        part.name,
        version=result.response.version,
        page_size=1000,
    ):
        names = [column.name for column in page.columns]
        rows.extend(dict(zip(names, row, strict=True)) for row in page.rows)
    assert len(rows) == part.source.row_count
    tables[part.name] = rows
print("Saved answer:", answer.response_id, "revision", answer.version)
print("Complete table rows:", {name: len(rows) for name, rows in tables.items()})

## Optional: execute the saved study again

This separate action creates a new Backend execution of the saved study using
the same period and costs. It does not rediscover data, rebuild the study,
refresh a saved roster or publish a new interpretation. It currently returns
execution status; response-plan replay remains separate work.

The execution ID below is independent of the research ID. Repeating the call
with the same ID attaches to the same execution.

In [ ]:
EXECUTION_ID = f"{PREFIX}-sp500_momentum_screener-{ATTEMPT}-execute"
assert result.reusable_study is not None
execution = await epoch.execute_saved(
    result.reusable_study,
    period=request.parameters.period,
    execution=request.parameters.execution,
    operation_id=EXECUTION_ID,
)
print("Execution operation:", EXECUTION_ID)
# Read this same ID later to get its current native status.
execution_status = await epoch.execution_operation(EXECUTION_ID)

## Close this client

Saved work remains on the services. Keep the readable research ID to reconnect.

In [ ]:
await epoch.aclose()